<a href="https://colab.research.google.com/github/abualjoud95/colab_repo/blob/main/nb/Qwen3_5_(2B)_FineTune_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [17]:
!pip install -q pandas scikit-learn datasets

In [18]:
import pandas as pd
from sklearn.model_selection import train_test_split

url = (
    "https://raw.githubusercontent.com/"
    "bazokhan/arabic-itsm-dataset/master/dataset_clean.csv"
)

df = pd.read_csv(url)

print("Original rows:", len(df))

# Remove exact duplicate tickets
df = df.drop_duplicates(
    subset=["title_ar", "description_ar"],
    keep="first"
)

# Remove rows missing essential fields
df = df.dropna(
    subset=[
        "title_ar",
        "description_ar",
        "category_level_1",
        "category_level_2",
        "category_level_3",
        "priority"
    ]
)

print("Clean rows:", len(df))
print(df["category_level_1"].value_counts())

Original rows: 10000
Clean rows: 9549
category_level_1
Access      1905
Software    1772
Hardware    1767
Network     1757
Service     1210
Security    1138
Name: count, dtype: int64


In [19]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["category_level_1"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["category_level_1"]
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 7639
Validation: 955
Test: 955


In [20]:
SYSTEM_PROMPT = """
You are an IT support ticket classification assistant.

Analyze the user's support ticket and return ONLY valid JSON.

The JSON must contain:
- category
- subcategory
- issue
- priority
- impact
- urgency
- sentiment
- tags
- summary

Use the exact classification labels provided by the training examples.
Do not invent information.
Do not include markdown or explanations outside JSON.
"""

In [21]:
import json

def make_example(row):
    ticket = f"""
Title: {row['title_ar']}

Description:
{row['description_ar']}
"""

    answer = {
        "category": row["category_level_1"],
        "subcategory": row["category_level_2"],
        "issue": row["category_level_3"],
        "priority": int(row["priority"]),
        "impact": int(row["impact"]),
        "urgency": int(row["urgency"]),
        "sentiment": row["sentiment"],
        "tags": row["tags"] if isinstance(row["tags"], list)
               else [],
        "summary": row["title_ar"]
    }

    return {
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": ticket
            },
            {
                "role": "assistant",
                "content": json.dumps(
                    answer,
                    ensure_ascii=False
                )
            }
        ]
    }

In [24]:
from datasets import Dataset

train_data = Dataset.from_list([
    make_example(row)
    for _, row in train_df.iterrows()
])

val_data = Dataset.from_list([
    make_example(row)
    for _, row in val_df.iterrows()
])

test_data = Dataset.from_list([
    make_example(row)
    for _, row in test_df.iterrows()
])

print(train_data[0]["messages"])

[{'content': "\nYou are an IT support ticket classification assistant.\n\nAnalyze the user's support ticket and return ONLY valid JSON.\n\nThe JSON must contain:\n- category\n- subcategory\n- issue\n- priority\n- impact\n- urgency\n- sentiment\n- tags\n- summary\n\nUse the exact classification labels provided by the training examples.\nDo not invent information.\nDo not include markdown or explanations outside JSON.\n", 'role': 'system'}, {'content': '\nTitle: الشاشة سودة\n\nDescription:\nالنت تقيل جداً النهاردة ومش عارف افتح اي حاجة، حد يشوف الموضوع ده. بقالها 11 ساعة تقريبا. لو محتاجين لوجز أو سكرينشوت أنا جاهز أبعتها.\n', 'role': 'user'}, {'content': '{"category": "Security", "subcategory": "Malware/Phishing", "issue": "Phishing Email", "priority": 3, "impact": 1, "urgency": 4, "sentiment": "neutral", "tags": [], "summary": "الشاشة سودة"}', 'role': 'assistant'}]


In [25]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3.5-2B",
    max_seq_length=max_seq_length,
    load_in_4bit=False,
    load_in_16bit=True,
    full_finetuning=False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    max_seq_length=max_seq_length,
)

==((====))==  Unsloth 2026.11.9: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


In [29]:
def format_example(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

train_data = train_data.map(format_example)
val_data = val_data.map(format_example)

print(train_data[0]["text"])

Map:   0%|          | 0/7639 [00:00<?, ? examples/s]

Map:   0%|          | 0/955 [00:00<?, ? examples/s]

<|im_start|>system
You are an IT support ticket classification assistant.

Analyze the user's support ticket and return ONLY valid JSON.

The JSON must contain:
- category
- subcategory
- issue
- priority
- impact
- urgency
- sentiment
- tags
- summary

Use the exact classification labels provided by the training examples.
Do not invent information.
Do not include markdown or explanations outside JSON.<|im_end|>
<|im_start|>user
Title: الشاشة سودة

Description:
النت تقيل جداً النهاردة ومش عارف افتح اي حاجة، حد يشوف الموضوع ده. بقالها 11 ساعة تقريبا. لو محتاجين لوجز أو سكرينشوت أنا جاهز أبعتها.<|im_end|>
<|im_start|>assistant
<think>

</think>

{"category": "Security", "subcategory": "Malware/Phishing", "issue": "Phishing Email", "priority": 3, "impact": 1, "urgency": 4, "sentiment": "neutral", "tags": [], "summary": "الشاشة سودة"}<|im_end|>



In [27]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_data,
    eval_dataset=val_data,
    args=SFTConfig(
        output_dir="crm_ticket_model",
        dataset_text_field="text",
        max_length=max_seq_length,

        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=4,

        num_train_epochs=2,
        learning_rate=1e-4,

        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,

        optim="adamw_8bit",
        weight_decay=0.01,
        warmup_ratio=0.03,

        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),

        seed=3407,
        report_to="none",
    ),
)

trainer_stats = trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"]:   0%|          | 0/7639 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"]:   0%|          | 0/955 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,639 | Num Epochs = 2 | Total steps = 3,820
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 10,911,744 of 2,224,153,408 (0.49% trained)


Epoch,Training Loss,Validation Loss
1,0.127003,0.127161
2,0.092088,0.104777


Unsloth: Restored added_tokens_decoder metadata in crm_ticket_model/checkpoint-1910/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in crm_ticket_model/checkpoint-3820/tokenizer_config.json.


In [28]:
model.save_pretrained("crm-ticket-lora")
tokenizer.save_pretrained("crm-ticket-lora")

Unsloth: Restored added_tokens_decoder metadata in crm-ticket-lora/tokenizer_config.json.


['crm-ticket-lora/processor_config.json']

In [30]:
from unsloth import FastLanguageModel

# Model A: Original
model_a, tokenizer_a = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3.5-2B",
    max_seq_length=2048,
    load_in_4bit=False,
)

# Model B: Fine-tuned
model_b, tokenizer_b = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3.5-2B",
    max_seq_length=2048,
    load_in_4bit=False,
)

model_b.load_adapter("crm-ticket-lora")

==((====))==  Unsloth 2026.11.9: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

==((====))==  Unsloth 2026.11.9: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

In [39]:

def analyze_ticket(model, processor, title, description):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"""
Title: {title}

Description:
{description}
"""
        }
    ]

    # Format the conversation without tokenizing it
    prompt = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Process the formatted text
    inputs = processor(
        text=[prompt],
        return_tensors="pt"
    )

    # Move tensors to model device
    inputs = {
        key: value.to(model.device) if hasattr(value, "to") else value
        for key, value in inputs.items()
    }

    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False
    )

    input_length = inputs["input_ids"].shape[1]

    response = processor.batch_decode(
        outputs[:, input_length:],
        skip_special_tokens=True
    )[0]

    return response


In [41]:
title = "مشكلة في تسجيل الدخول"

description = """
لا أستطيع الدخول إلى حسابي منذ هذا الصباح.
عند إدخال كلمة المرور يطلب مني رمز MFA،
ولكن الرمز لا يعمل.
"""

original_result = analyze_ticket(
    model_a,
    tokenizer_a,
    title,
    description
)

trained_result = analyze_ticket(
    model_b,
    tokenizer_b,
    title,
    description
)

print("========== ORIGINAL MODEL ==========")
print(original_result)

print("\n========== FINE-TUNED MODEL ==========")
print(trained_result)

OutOfMemoryError: CUDA out of memory. Tried to allocate 976.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 911.81 MiB is free. Including non-PyTorch memory, this process has 13.67 GiB memory in use. Of the allocated memory 13.04 GiB is allocated by PyTorch, and 78.09 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [42]:
print("ORIGINAL MODEL RESULT:")
print(original_result)

ORIGINAL MODEL RESULT:
{
  "category": "Authentication",
  "subcategory": "MFA Failure",
  "issue": "User cannot log in due to MFA code failure",
  "priority": "High",
  "impact": "User unable to access account",
  "urgency": "High",
  "sentiment": "Negative",
  "tags": ["login", "mfa", "authentication", "security"],
  "summary": "User reports inability to log in after MFA code failed to generate."
}



In [43]:
import gc
import torch

del model_a
del tokenizer_a

gc.collect()
torch.cuda.empty_cache()

print(
    "Available GPU memory:",
    round(
        torch.cuda.mem_get_info()[0] / 1024**3,
        2
    ),
    "GB"
)

Available GPU memory: 6.99 GB


In [44]:
trained_result = analyze_ticket(
    model_b,
    tokenizer_b,
    title,
    description
)

print("FINE-TUNED MODEL:")
print(trained_result)

InternalTorchDynamoError: ValueError: stoi


In [45]:
import torch

torch._dynamo.reset()

torch._dynamo.config.disable = True

print("TorchDynamo disabled")

TorchDynamo disabled


In [46]:

def analyze_ticket(model, processor, title, description):

    model.eval()

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"""
Title: {title}

Description:
{description}
"""
        }
    ]

    prompt = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=[prompt],
        return_tensors="pt"
    )

    inputs = {
        key: value.to(model.device)
        if hasattr(value, "to") else value
        for key, value in inputs.items()
    }

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            use_cache=True
        )

    input_length = inputs["input_ids"].shape[1]

    response = processor.batch_decode(
        outputs[:, input_length:],
        skip_special_tokens=True
    )[0]

    return response


In [47]:
trained_result = analyze_ticket(
    model_b,
    tokenizer_b,
    title,
    description
)

print(trained_result)

RuntimeError: Expected all tensors to be on the same device, but got mat2 is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA_mm)

In [48]:
for name, param in model_b.named_parameters():
    if "lora_" in name:
        print(
            name,
            "Device:", param.device,
            "Dtype:", param.dtype
        )

model.language_model.layers.0.mlp.gate_proj.lora_A.default.weight Device: cpu Dtype: torch.float16
model.language_model.layers.0.mlp.gate_proj.lora_B.default.weight Device: cpu Dtype: torch.float16
model.language_model.layers.0.mlp.up_proj.lora_A.default.weight Device: cpu Dtype: torch.float16
model.language_model.layers.0.mlp.up_proj.lora_B.default.weight Device: cpu Dtype: torch.float16
model.language_model.layers.0.mlp.down_proj.lora_A.default.weight Device: cpu Dtype: torch.float16
model.language_model.layers.0.mlp.down_proj.lora_B.default.weight Device: cpu Dtype: torch.float16
model.language_model.layers.1.mlp.gate_proj.lora_A.default.weight Device: cpu Dtype: torch.float16
model.language_model.layers.1.mlp.gate_proj.lora_B.default.weight Device: cpu Dtype: torch.float16
model.language_model.layers.1.mlp.up_proj.lora_A.default.weight Device: cpu Dtype: torch.float16
model.language_model.layers.1.mlp.up_proj.lora_B.default.weight Device: cpu Dtype: torch.float16
model.language_mod

In [49]:
model_b = model_b.to("cuda")

for name, param in model_b.named_parameters():
    if "lora_" in name:
        print(name, param.device)

RuntimeError: You can't move a model that has some modules offloaded to cpu or disk.